In [1]:
from pynq import Overlay

ol = Overlay("axi_checker.bit")
ol?

In [2]:
checker = ol.axi_checker_top_0
print("base address: 0x%08X" % checker.mmio.base_addr)
print("address range: 0x%X" % checker.mmio.length)

base address: 0x40000000
address range: 0x1000


In [3]:
status = checker.mmio.read(0x04)
print("STATUS = 0x%08X" % status)
print("  empty    =", (status >> 0) & 1)
print("  full     =", (status >> 1) & 1)
print("  overflow =", (status >> 2) & 1)
print("  busy     =", (status >> 3) & 1)

STATUS = 0x00000001
  empty    = 1
  full     = 0
  overflow = 0
  busy     = 0


In [4]:
# --- register map ---
CTRL    = 0x00   # bits[7:4]=fault_mode, bit0=start
STATUS  = 0x04
COUNT   = 0x08
EV_TS   = 0x0C
EV_VIOL = 0x10
EV_POP  = 0x14

import time

def run_fault(mode):
    """Arm a fault mode and start one transaction."""
    checker.mmio.write(CTRL, (mode << 4) | 1)
    # wait for the rogue master to finish (busy bit clears)
    for _ in range(100):
        if not (checker.mmio.read(STATUS) >> 3) & 1:
            break
        time.sleep(0.001)

# fire a legal transaction, then a C01-on-AW fault
run_fault(0)   # legal
run_fault(1)   # C01 retract on AW

print("events buffered:", checker.mmio.read(COUNT))

events buffered: 2


In [5]:
VIOL_NAMES = {
    0:  "C01 AW-retract",   1:  "C01 W-retract",   2:  "C01 B-retract",
    3:  "C01 AR-retract",   4:  "C01 R-retract",   5:  "C02 AW-stable",
    6:  "C02 W-stable",     7:  "C02 B-stable",    8:  "C02 AR-stable",
    9:  "C02 R-stable",     10: "C03 reset",       11: "C05 EXOKAY",
    12: "C07 write-order",  13: "C08 read-order",  14: "C09 align",
}

def decode(vec):
    """Turn a 15-bit violation vector into readable check names."""
    return [name for bit, name in VIOL_NAMES.items() if (vec >> bit) & 1]

def drain():
    """Pop every buffered event and return it as a list of dicts."""
    events = []
    while not (checker.mmio.read(STATUS) & 1):      # while not empty
        ts   = checker.mmio.read(EV_TS)             # read head...
        viol = checker.mmio.read(EV_VIOL)
        checker.mmio.write(EV_POP, 1)               # ...then acknowledge
        events.append({"ts": ts, "viol": viol, "checks": decode(viol)})
    return events

for i, e in enumerate(drain(), 1):
    print(f"EVENT {i}  ts={e['ts']:>8}  {', '.join(e['checks'])}")

EVENT 1  ts=4196148971  C01 AW-retract
EVENT 2  ts=4196148987  C01 AW-retract


In [6]:
def drain_relative():
    events = drain()
    if not events:
        return []
    t0 = events[0]["ts"]
    for e in events:
        e["us"] = (e["ts"] - t0) / 50.0     # 50 MHz -> microseconds
    return events

# test all fault modes
FAULT_NAMES = {0: "legal", 1: "C01 on AW", 2: "C02 on AW",
               3: "C01 on W", 4: "C02 on W", 5: "C09 misaligned"}

for mode in range(6):
    run_fault(mode)

evts = drain_relative()
print(f"{len(evts)} events captured\n")
for i, e in enumerate(evts, 1):
    print(f"{i:>2}.  +{e['us']:>8.2f} us   {', '.join(e['checks'])}")

7 events captured

 1.  +    0.00 us   C01 AW-retract
 2.  +    0.32 us   C01 AW-retract
 3.  +   34.82 us   C02 AW-stable
 4.  +   68.56 us   C01 W-retract
 5.  +   68.88 us   C01 W-retract
 6.  +  100.50 us   C02 W-stable
 7.  +  131.66 us   C09 align


In [8]:
# ============================================================
#  AXI4-Lite Protocol Checker - live dashboard
# ============================================================
import time, threading, random
from io import BytesIO
from collections import Counter

import ipywidgets as widgets
from IPython.display import display

import matplotlib
matplotlib.use("Agg")          # off-screen render; we push PNGs to a widget
import matplotlib.pyplot as plt

# ---------------- register map ----------------
CTRL, STATUS, COUNT, EV_TS, EV_VIOL, EV_POP = 0x00, 0x04, 0x08, 0x0C, 0x10, 0x14

CLK_HZ  = 50_000_000
TS_BITS = 32
TS_WRAP = 1 << TS_BITS          # counter rolls over every ~86 s at 50 MHz

VIOL_NAMES = {
    0:  "C01 AW-retract",   1:  "C01 W-retract",   2:  "C01 B-retract",
    3:  "C01 AR-retract",   4:  "C01 R-retract",   5:  "C02 AW-stable",
    6:  "C02 W-stable",     7:  "C02 B-stable",    8:  "C02 AR-stable",
    9:  "C02 R-stable",     10: "C03 reset",       11: "C05 EXOKAY",
    12: "C07 write-order",  13: "C08 read-order",  14: "C09 align",
}
LANES = [VIOL_NAMES[i] for i in range(15)]          # fixed y-axis order

FAULT_MODES = {
    "Legal traffic (no fault)": 0,
    "C01 - retract AWVALID":    1,
    "C02 - mutate AWADDR":      2,
    "C01 - retract WVALID":     3,
    "C02 - mutate WDATA":       4,
    "C09 - misaligned address": 5,
}
FAMILY_COLOR = {"C01": "#c0392b", "C02": "#d68910", "C03": "#7d3c98",
                "C05": "#1a5276", "C07": "#117864", "C08": "#0e6655",
                "C09": "#873600"}

# ---------------- hardware access ----------------
def run_fault(mode, timeout=0.2):
    """Arm a fault mode and start one transaction; wait for busy to clear."""
    checker.mmio.write(CTRL, (mode << 4) | 1)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if not (checker.mmio.read(STATUS) >> 3) & 1:
            return
        time.sleep(0.001)

# ---- timestamp wrap tracking ----------------------------------
_wraps, _last_raw = 0, None

def _absolute(ts_raw):
    """Extend the 32-bit hardware counter into a monotonic value."""
    global _wraps, _last_raw
    if _last_raw is not None and ts_raw < _last_raw:
        _wraps += 1                      # counter rolled over
    _last_raw = ts_raw
    return ts_raw + _wraps * TS_WRAP

def decode(vec):
    return [n for b, n in VIOL_NAMES.items() if (vec >> b) & 1]

def drain():
    """Pop every buffered event: read head, THEN acknowledge."""
    out = []
    while not (checker.mmio.read(STATUS) & 1):        # while not empty
        ts   = checker.mmio.read(EV_TS)
        viol = checker.mmio.read(EV_VIOL)
        checker.mmio.write(EV_POP, 1)
        out.append({"cyc": _absolute(ts), "viol": viol, "checks": decode(viol)})
    return out

# ---------------- dashboard state ----------------
events   = []                 # every event, with t_sec relative to the first
counts   = Counter()
t0_cyc   = None
_running = False
_thread  = None

def _ingest(new):
    global t0_cyc
    for e in new:
        if t0_cyc is None:
            t0_cyc = e["cyc"]
        e["t_sec"] = (e["cyc"] - t0_cyc) / CLK_HZ
        events.append(e)
        for c in e["checks"]:
            counts[c] += 1

# ---------------- widgets ----------------
mode_sel   = widgets.Dropdown(options=list(FAULT_MODES.keys()),
                              layout=widgets.Layout(width="260px"))
inject_btn = widgets.Button(description="Inject", button_style="danger", icon="bolt")
auto_tog   = widgets.ToggleButton(value=False, description="Live stream",
                                  button_style="info", icon="play")
rate_sl    = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1,
                                 description="interval s", readout_format=".1f",
                                 layout=widgets.Layout(width="260px"))
clear_btn  = widgets.Button(description="Reset")
metrics    = widgets.HTML()
canvas     = widgets.Image(format="png", layout=widgets.Layout(width="900px"))

def _metric_box(label, value, color="#2c3e50"):
    return (f"<div style='display:inline-block;min-width:120px;padding:6px 14px;"
            f"margin-right:8px;border-left:4px solid {color};background:#f7f7f7'>"
            f"<div style='font-size:11px;color:#666'>{label}</div>"
            f"<div style='font-size:20px;font-weight:600;color:{color}'>{value}</div></div>")

def refresh_metrics():
    s   = checker.mmio.read(STATUS)
    ovf = bool((s >> 2) & 1)
    span = events[-1]["t_sec"] if events else 0.0
    rate = (len(events) / span) if span > 0 else 0.0
    metrics.value = (
        _metric_box("EVENTS", len(events)) +
        _metric_box("DISTINCT CHECKS", len(counts)) +
        _metric_box("RATE /s", f"{rate:,.1f}") +
        _metric_box("FIFO", "empty" if s & 1 else "data", "#117864") +
        _metric_box("OVERFLOW", "YES" if ovf else "no",
                    "#c0392b" if ovf else "#117864")
    )

def refresh_plot():
    fig, (ax_t, ax_b) = plt.subplots(
        2, 1, figsize=(9, 5.2), gridspec_kw={"height_ratios": [3, 2]})

    # ---- top: violation timeline (one lane per check) ----
    ax_t.set_title("Protocol violations over time", fontsize=11, loc="left")
    for i, lane in enumerate(LANES):
        ax_t.axhline(i, color="#eeeeee", lw=1, zorder=0)
    if events:
        for e in events:
            for c in e["checks"]:
                ax_t.scatter(e["t_sec"], LANES.index(c), s=48,
                             color=FAMILY_COLOR[c[:3]], zorder=3,
                             edgecolors="white", linewidths=0.6)
        xmax = max(1.0, events[-1]["t_sec"] * 1.05)
    else:
        xmax = 1.0
    ax_t.set_yticks(range(len(LANES)))
    ax_t.set_yticklabels(LANES, fontsize=7)
    ax_t.set_xlim(-xmax * 0.02, xmax)
    ax_t.set_ylim(-0.5, len(LANES) - 0.5)
    ax_t.set_xlabel("seconds since first violation", fontsize=8)
    for sp in ("top", "right"):
        ax_t.spines[sp].set_visible(False)

    # ---- bottom: counts per check ----
    ax_b.set_title("Violations by check", fontsize=11, loc="left")
    if counts:
        names = sorted(counts, key=lambda n: counts[n])
        ax_b.barh(names, [counts[n] for n in names],
                  color=[FAMILY_COLOR[n[:3]] for n in names])
        for i, n in enumerate(names):
            ax_b.text(counts[n], i, f" {counts[n]}", va="center", fontsize=8)
        ax_b.set_xlim(0, max(counts.values()) * 1.18)
    ax_b.tick_params(labelsize=7)
    for sp in ("top", "right"):
        ax_b.spines[sp].set_visible(False)

    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    canvas.value = buf.getvalue()

def refresh_all():
    refresh_metrics()
    refresh_plot()

# ---------------- callbacks ----------------
def on_inject(_):
    run_fault(FAULT_MODES[mode_sel.value])
    _ingest(drain())
    refresh_all()

def on_clear(_):
    global t0_cyc
    drain()                      # flush hardware FIFO too
    events.clear(); counts.clear(); t0_cyc = None
    refresh_all()

def _worker():
    """Background loop: inject a random fault, drain, redraw."""
    while _running:
        mode = random.choice([0, 1, 2, 3, 4, 5])   # bias includes legal traffic
        run_fault(mode)
        new = drain()
        if new:
            _ingest(new)
        refresh_all()
        time.sleep(rate_sl.value)

def on_auto(change):
    global _running, _thread
    if change["new"]:
        _running = True
        auto_tog.description, auto_tog.icon = "Stop", "stop"
        _thread = threading.Thread(target=_worker, daemon=True)
        _thread.start()
    else:
        _running = False
        auto_tog.description, auto_tog.icon = "Live stream", "play"

inject_btn.on_click(on_inject)
clear_btn.on_click(on_clear)
auto_tog.observe(on_auto, names="value")

display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:2px'>AXI4-Lite Protocol Checker</h3>"
                 "<div style='color:#666;font-size:12px;margin-bottom:10px'>"
                 "live protocol health &mdash; PYNQ-Z2 / Zynq-7020 @ 50 MHz</div>"),
    widgets.HBox([mode_sel, inject_btn, auto_tog, rate_sl, clear_btn]),
    metrics,
    canvas,
]))
refresh_all()